# Task 3 — Gender MixUp 0.20 refit

Train **one fresh model on all 32,773 development images**, using the same
**MixUp 0.20 recipe as the five-fold runner**, for **30 epochs** with cosine **T_max=30**.
Keep the corrected Gender labels, GeM CNN, dropout and image transforms. **No SAM.**

Use a fresh **Colab L4 GPU**. The new notebook and source file must be pushed to the
GitHub branch below before **Run all**. Reuse the existing teacher ZIP in Drive.
The refit saves to its own folder and does not replace the accepted final Gender model.


## 1. Mount Drive and select the repository

In [1]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task3-mixup-five-fold-training"
REPO_DIR = Path("/content/MLA2")
DRIVE_PROJECT = Path("/content/drive/MyDrive/MLA2")
DATA_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
drive.mount("/content/drive", force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Fetch code from GitHub

Keep local edits safe: repository updates use a fast-forward merge.

In [2]:
def run_checked(command):
    return subprocess.run([str(x) for x in command], check=True)


if (REPO_DIR / ".git").is_dir():
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError("The local checkout belongs to another repository")
    run_checked(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "switch", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
else:
    run_checked(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:")
run_checked(["git", "rev-parse", "HEAD"])

Code commit:


CompletedProcess(args=['git', 'rev-parse', 'HEAD'], returncode=0)

## 3. Reuse the existing teacher images

Read the canonical development rows. Extract only missing training images.
The archive cannot overwrite Git code or the saved split. The refit preflight
then checks every development image against its recorded hash.

In [3]:
import pandas as pd

splits = pd.read_csv(REPO_DIR / "data/processed/splits.csv", keep_default_na=False)
paths = splits.loc[splits.partition.eq("development"), "path"].tolist()
missing = [source_path for source_path in paths if not (REPO_DIR / source_path).is_file()]
if missing:
    local_zip = Path("/content/task3-data.zip")
    shutil.copyfile(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        for source_path in missing:
            if not source_path.startswith("data/raw/teacher/train/images_train/"):
                raise ValueError(f"Unexpected development image path: {source_path}")
            target = (REPO_DIR / source_path).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError("Image path leaves the repository")
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(target.suffix + ".partial")
            with archive.open(source_path) as source, partial.open("wb") as output:
                shutil.copyfileobj(source, output)
            partial.replace(target)
print(f"Development images ready: {len(paths):,}")

Development images ready: 32,773


## 4. Check the fixed recipe and Colab runtime

SmallCNN + GeM p=3, **390,181 parameters**, dropout **0.30**, MixUp **0.20**, no SAM.
AdamW: learning rate **0.001**, minimum **0.00001**, weight decay **0.0001**, batch **128**, seed **2753**.
Keep translation ±2 px (50%), mild darkening (25%) and grayscale (10%).
Fit normalization on all development images. There is no validation split or early stopping.

Match the recorded Colab stack in `requirements/colab-task3-runtime.txt`, as in the five-fold run.
The check stops before training if the runtime differs. Do not install the local project's
newer PyTorch requirements over Colab's CUDA build.


In [4]:
import json

import torch

from fashion.train.task3_baseline import runtime_environment, validate_verified_colab_runtime
from fashion.train.task3_gender_mixup_refit import EXPERIMENT, SOURCE_CONFIG

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab L4 GPU runtime before training")
validate_verified_colab_runtime(runtime_environment(torch.device("cuda")))
saved = json.loads((REPO_DIR / SOURCE_CONFIG).read_text())
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
print("Epochs:", saved["epochs"], "Cosine T_max:", saved["epochs"], "SAM: disabled")
print("Class labels: Boys, Girls, Men, Unisex, Women")
print("Output:", DRIVE_TASK_DIR / "experiments" / EXPERIMENT / "gender")


GPU: NVIDIA A100-SXM4-40GB PyTorch: 2.11.0+cu128
Epochs: 30 Cosine T_max: 30 SAM: disabled
Class labels: Boys, Girls, Men, Unisex, Women
Output: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_refit/gender


## 5. Train and save the refit

The runner checks the source recipe, saved folds, corrected labels and every development image.
It registers the run before the first update, then trains fresh weights for **30 epochs**.
Each epoch records mixed training loss and checks that every development row was used once.

Repeat Run all verifies and reuses a completed refit. An interrupted fit starts again from scratch
in a new folder. Keep only one active Colab session writing this refit experiment.
The refit does not need completed five-fold checkpoints and never loads them.


In [5]:
from fashion.train.task3_gender_mixup_refit import run_gender_mixup_refit

result = run_gender_mixup_refit(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
)
print("Status:", result["status"])
print("Reused:", result["reused"])
print("Model manifest:", result["manifest_path"])

Checking 32,773 development image hashes
Epoch 1/30: mixed training loss 0.7113
Epoch 2/30: mixed training loss 0.5943
Epoch 3/30: mixed training loss 0.5740
Epoch 4/30: mixed training loss 0.5293
Epoch 5/30: mixed training loss 0.5242
Epoch 6/30: mixed training loss 0.5003
Epoch 7/30: mixed training loss 0.4958
Epoch 8/30: mixed training loss 0.4797
Epoch 9/30: mixed training loss 0.4822
Epoch 10/30: mixed training loss 0.4489
Epoch 11/30: mixed training loss 0.4589
Epoch 12/30: mixed training loss 0.4394
Epoch 13/30: mixed training loss 0.4205
Epoch 14/30: mixed training loss 0.4216
Epoch 15/30: mixed training loss 0.4181
Epoch 16/30: mixed training loss 0.4202
Epoch 17/30: mixed training loss 0.4200
Epoch 18/30: mixed training loss 0.4143
Epoch 19/30: mixed training loss 0.4039
Epoch 20/30: mixed training loss 0.3762
Epoch 21/30: mixed training loss 0.3969
Epoch 22/30: mixed training loss 0.3690
Epoch 23/30: mixed training loss 0.3960
Epoch 24/30: mixed training loss 0.3452
Epoch 25

## 6. Check the saved training record

Mixed training losses are not validation scores. Use the five-fold notebook for model comparison.
This refit is a separate candidate artifact: running it does not mark MixUp as the winner.
No holdout/test predictions, submission file or change to the accepted final model is made here.

`model_manifest.json` links the checkpoint, normalization, class order and file hashes.
For later inference on new images, use this one model and its saved normalization,
apply softmax, then choose the class with the largest probability.


In [6]:
manifest_dir = Path(result["manifest_path"]).parent
history_path = manifest_dir / result["files"]["history.csv"]["path"]
history = pd.read_csv(history_path)
assert history.epoch.tolist() == list(range(1, 31))
assert history.training_rows.eq(32773).all()
assert history.selected_checkpoint.tolist() == [False] * 29 + [True]
display(history.tail())
print("Checkpoint:", manifest_dir / result["files"]["final_epoch.pt"]["path"])
print("Normalization:", manifest_dir / result["files"]["normalization.json"]["path"])

,epoch,learning_rate,train_mixed_loss,training_rows,optimizer_steps,selected_checkpoint
25,26,0.000076,0.381700,32773,257,False
26,27,0.000053,0.338135,32773,257,False
27,28,0.000034,0.350543,32773,257,False
28,29,0.000021,0.356163,32773,257,False
29,30,0.000013,0.343908,32773,257,True


Checkpoint: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_refit/gender/t3_gender_name_truth_mixup_alpha020_refit_20260911T041436Z_3af0b94e/final_epoch.pt
Normalization: /content/drive/MyDrive/MLA2/task3/experiments/t3_gender_name_truth_mixup_alpha020_refit/gender/t3_gender_name_truth_mixup_alpha020_refit_20260911T041436Z_3af0b94e/normalization.json
